In [ ]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.estimator import Estimator
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer
import boto3
import pandas as pd

sess = sagemaker.Session()
role = get_execution_role()

region = sess.boto_session.region_name
account = boto3.client("sts").get_caller_identity()["Account"]
bucket = sess.default_bucket()

print("region:", region)
print("account:", account)
print("bucket:", bucket)
print("role:", role)

In [11]:
!cat ../container/Dockerfile

FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV PATH="/opt/program/.venv/bin:/opt/program:${PATH}"

WORKDIR /opt/program

RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    libgomp1 \
    && rm -rf /var/lib/apt/lists/*

RUN pip install --no-cache-dir uv

COPY pyproject.toml uv.lock README.md /opt/program/
RUN uv sync --frozen --no-dev --no-install-project

COPY src /opt/program/src
COPY container/train /opt/program/train
COPY container/serve /opt/program/serve
COPY container/predictor.py /opt/program/predictor.py
COPY container/wsgi.py /opt/program/wsgi.py

RUN chmod +x /opt/program/train /opt/program/serve

In [13]:
%%bash
set -euo pipefail

IMAGE="predict-future-sales-byoc"
ACCOUNT=$(aws sts get-caller-identity --query Account --output text)

REGION="${AWS_REGION:-${AWS_DEFAULT_REGION:-}}"
if [ -z "$REGION" ]; then
  REGION=$(aws configure get region || true)
fi
if [ -z "$REGION" ]; then
  REGION="us-east-1"
fi

echo "ACCOUNT=$ACCOUNT"
echo "REGION=$REGION"

FULLNAME="${ACCOUNT}.dkr.ecr.${REGION}.amazonaws.com/${IMAGE}:latest"
echo "FULLNAME=$FULLNAME"

chmod +x ../container/train ../container/serve

aws ecr describe-repositories --region "$REGION" --repository-names "$IMAGE" >/dev/null 2>&1 || \
  aws ecr create-repository --region "$REGION" --repository-name "$IMAGE" >/dev/null

aws ecr get-login-password --region "$REGION" | \
  docker login --username AWS --password-stdin "${ACCOUNT}.dkr.ecr.${REGION}.amazonaws.com"

docker build --network sagemaker -t "${IMAGE}:latest" -f ../container/Dockerfile ..

docker tag "${IMAGE}:latest" "$FULLNAME"
docker push "$FULLNAME"

echo "PUSHED_IMAGE_URI=$FULLNAME"

ACCOUNT=494321812137
REGION=us-east-1
FULLNAME=494321812137.dkr.ecr.us-east-1.amazonaws.com/predict-future-sales-byoc:latest


WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores



Login Succeeded


DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.



Sending build context to Docker daemon  985.1kB
Step 1/16 : FROM python:3.12-slim
 ---> 6f90d4a79e7a
Step 2/16 : ENV PYTHONDONTWRITEBYTECODE=1
 ---> Using cache
 ---> 9f69481e277e
Step 3/16 : ENV PYTHONUNBUFFERED=1
 ---> Using cache
 ---> 37b22a55231c
Step 4/16 : ENV PATH="/opt/program/.venv/bin:/opt/program:${PATH}"
 ---> Using cache
 ---> c8415719d947
Step 5/16 : WORKDIR /opt/program
 ---> Using cache
 ---> 609209981f9c
Step 6/16 : RUN apt-get update && apt-get install -y --no-install-recommends     build-essential     libgomp1     && rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> c5ba9610a14e
Step 7/16 : RUN pip install --no-cache-dir uv
 ---> Using cache
 ---> bb5efff85b12
Step 8/16 : COPY pyproject.toml uv.lock README.md /opt/program/
 ---> Using cache
 ---> d6bf8ea62a39
Step 9/16 : RUN uv sync --frozen --no-dev --no-install-project
 ---> Using cache
 ---> b6e199cf63c1
Step 10/16 : COPY src /opt/program/src
 ---> Using cache
 ---> 6f2057ee6eed
Step 11/16 : COPY container/trai

In [14]:
# S3 prefix 
prefix = "predict-future-sales/byoc"

import boto3
import re
import os
import numpy as np
import pandas as pd
from sagemaker import get_execution_role

role = get_execution_role()
role

'arn:aws:iam::494321812137:role/SageMakerStudioExecutionRole2026'

In [15]:
import sagemaker as sage
from time import gmtime, strftime

sess = sage.Session()

In [16]:
WORK_DIRECTORY = "../data/prep"   

default_bucket = sess.default_bucket()
default_bucket_prefix = sess.default_bucket_prefix

# Si hay prefix default del bucket, lo anexamos como en el ejemplo
if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

data_location = sess.upload_data(WORK_DIRECTORY, bucket=default_bucket, key_prefix=f"{prefix}/input/prep")

default_bucket, data_location

('sagemaker-us-east-1-494321812137',
 's3://sagemaker-us-east-1-494321812137/predict-future-sales/byoc/input/prep')

In [18]:
account = sess.boto_session.client("sts").get_caller_identity()["Account"]
region = sess.boto_session.region_name

image_name = "predict-future-sales-byoc"
image = f"{account}.dkr.ecr.{region}.amazonaws.com/{image_name}:latest"
image

'494321812137.dkr.ecr.us-east-1.amazonaws.com/predict-future-sales-byoc:latest'

In [19]:
s3_output_path = f"s3://{default_bucket}/{prefix}/output"

tree = sage.estimator.Estimator(
    image,
    role,
    1,
    "ml.m5.large",
    output_path=s3_output_path,
    sagemaker_session=sess,
)

# contenedor espera el channel "prep"
tree.fit({"prep": data_location})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: predict-future-sales-byoc-2026-03-10-01-13-46-270


2026-03-10 01:13:47 Starting - Starting the training job...
2026-03-10 01:14:03 Starting - Preparing the instances for training...
2026-03-10 01:14:26 Downloading - Downloading input data...
2026-03-10 01:15:06 Training - Training image download completed. Training in progress..2026-03-10 01:15:17,343 - src.training.train - INFO - Iniciando entrenamiento.
2026-03-10 01:15:20,012 - src.training.train - INFO - Datasets cargados. train_rows=7068600 valid_rows=214200
2026-03-10 01:15:20,018 - src.training.train - INFO - Features=47 | cat_features=4
2026-03-10 01:15:20,019 - src.training.train - INFO - Entrenando clasificador (venta vs no venta).
2026-03-10 01:22:34,895 - src.training.train - INFO - Clasificador entrenado. prob_valid_mean=0.1109
2026-03-10 01:22:34,895 - src.training.train - INFO - Entrenando regresor (unidades | venta).

2026-03-10 01:23:28 Uploading - Uploading generated training model2026-03-10 01:23:23,233 - src.training.train - INFO - Regresor entrenado. mu_valid_mean=

In [20]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor = tree.deploy(
    1,
    "ml.m5.large",
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
)

INFO:sagemaker:Creating model with name: predict-future-sales-byoc-2026-03-10-01-24-58-743
INFO:sagemaker:Creating endpoint-config with name predict-future-sales-byoc-2026-03-10-01-24-58-743
INFO:sagemaker:Creating endpoint with name predict-future-sales-byoc-2026-03-10-01-24-58-743


-----!

In [26]:
#estado del endpoint
sm = boto3.client("sagemaker")
desc = sm.describe_endpoint(EndpointName=predictor.endpoint_name)

print("Endpoint status:", desc["EndpointStatus"])
print("Creation time:", desc["CreationTime"])
print("Endpoint ARN:", desc["EndpointArn"])

Endpoint status: InService
Creation time: 2026-03-10 01:25:00.370000+00:00
Endpoint ARN: arn:aws:sagemaker:us-east-1:494321812137:endpoint/predict-future-sales-byoc-2026-03-10-01-24-58-743


In [25]:
sample = pd.read_parquet("../data/prep/test_features.parquet").head(5)
payload = sample.to_dict(orient="records")

resp = predictor.predict(payload)
resp

{'predictions': [0.09952517598867416,
  0.23363599181175232,
  0.24812012910842896,
  0.2972855269908905,
  0.04560829699039459]}

In [27]:
sess.delete_endpoint(predictor.endpoint_name)

INFO:sagemaker:Deleting endpoint with name: predict-future-sales-byoc-2026-03-10-01-24-58-743
